# NLLB-200 other-languages: combined model, v3 (clean monitoring split)

**Combined/multilingual only - independent per-language training is not repeated here.**

**What v1 got wrong** (`06_nllb_other_languages.ipynb`, `train.py`): its combined model
regressed Somali and Dholuo below their own zero-shot baseline (chrF2++ -2.29 and -4.21)
despite Kiswahili improving (+1.34). Two real causes: `train.py`'s internal evaluation
never sets `forced_bos_token_id`, beam search, or anti-repetition guards (confirmed by
reading the file directly), so for a three-target-language validation set the in-training
"Chrf2pp" number had no reliable way to know which language a row should even decode
into - almost certainly why it never resembled any real per-language score. And the
checkpoint-selection metric was aggregated across all three languages at once, so it
could pick a step that was still improving Kiswahili while already hurting Somali/Dholuo.

**What v2 fixed, and what it got wrong** (`06_nllb_other_languages_v2.ipynb`,
`train_other_langs_v2.py`): replaced the Trainer's own eval with real, correctly-
configured per-language generation (explicit `forced_bos_token_id`, beam=4,
anti-repetition guards), lowered the learning rate 5e-5 -> 1e-5, and added a warmup
schedule - reasoning that Somali/Dholuo already have strong pretrained NLLB capability
(unlike Ekegusii, which needed a brand-new token bootstrapped from nothing), so
regressing them is a forgetting risk from an already-good baseline, not an underfitting
problem. A reviewed third-party suggestion to raise LoRA rank to 32 and expand to
q/k/v/out_proj was rejected on the same reasoning - more capacity is the wrong direction
for a model that's already regressing, not underfitting. This shrank the regression
substantially (Somali -2.29 -> -0.42, Dholuo -4.21 -> -0.64) - real progress, not fully
closed. The reason: v2 used `shared_val` (05's validation output) for per-language
monitoring, and `shared_val` turned out to have **86.5% near-duplicate overlap with the
training pool** - nearly as contaminated as the 78% that made the old, pre-v1 naive-split
comparison notebook's numbers untrustworthy in the first place. `validation_psa.csv` was
built for the original Ekegusii curriculum pipeline, where some train/validation overlap
is normal - it was never held out to the standard `test_psa.csv` was. So v2's "best step"
(500, the very first checkpoint) was chosen by a signal that rewarded memorization, not
generalization - confirmed by the ~12-18 point gap between what that checkpoint scored
during v2's training (Somali 85.07, Dholuo 77.80) and what it actually scored on the
clean, held-out final test set (73.07, 59.56).

**What v3 (this notebook) fixes**: the already-confirmed-clean 985-row `shared_test` pool
(5% overlap with training, not 86.5%) is split ONCE, stratified by domain, into a
**monitor** subset (per-language stopping decisions during training) and a **final**
subset (touched only once, at the very end) - never `shared_val` for anything chrF-based
again. Eval frequency also dropped 500 -> 100 steps, since every protected language's
score peaked at the very first checkpoint v2 measured and declined after, meaning the
true peak - now measured against a clean signal - could easily be earlier than 500 steps
too. `patience` scaled 3 -> 5 to compensate for the finer granularity without becoming
twitchy on single-check noise.

**Still unchanged from v1/v2, deliberately**: LoRA rank (16), target modules
(q_proj/v_proj only) - more adapter capacity is the right move for an underfitting model
(Ekegusii's situation), not one that's regressing on a small, narrow, single-domain
dataset. Also not attempted: mixing general-domain (non-PSA) data into training - this
project's own Ekegusii ablation found "mixed" beat narrow domain-only training, but it
needs data this project doesn't currently have staged for these three languages.

**New files, nothing overwritten**: `train_other_langs_v3.py` (project root, alongside
`train.py`/`train_other_langs_v2.py` - not a modification of either) does the actual
training/eval/stopping; `06_nllb_other_languages_v3.ipynb` (this file) is new alongside
`06_nllb_other_languages.ipynb` and `06_nllb_other_languages_v2.ipynb`.
`arch_config.py`/`metrics.py`/`inference.py`/`peft_setup.py` are reused as-is via import,
unmodified. All checkpoint/log/deployment names use a `_v3` suffix so nothing here can
collide with or overwrite v1's or v2's outputs.

**Data**: `05_other_languages_preprocess.ipynb`'s three outputs, unchanged - same shared
985-row test set, same training pool.

## 1. Setup

In [1]:
!pip install -q torch transformers accelerate sentencepiece evaluate sacrebleu mlflow peft matplotlib pandas datasets


### GPU / CUDA check — run this before anything else

In [2]:
import torch
print(f"torch version       : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU visible to torch - fix this before continuing.")


torch version       : 2.11.0+cu128
CUDA available      : True
GPU                 : NVIDIA H100 PCIe


In [3]:
import sys, os
from pathlib import Path

def find_project_root(marker="arch_config.py"):
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} by walking up from {current}")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"project root: {PROJECT_ROOT}")
if "final_training" not in str(PROJECT_ROOT).lower().replace(" ", "_"):
    print("\n  WARNING: 'Final_Training' doesn't appear in PROJECT_ROOT above - move "
          "this notebook under Final_Training/Notebooks/ before continuing.")

# Reused as-is - just path conventions, unrelated to what changed in this retrain.
from arch_config import CHECKPOINTS_DIR, LOGS_DIR, DEPLOYMENT_DIR, DATA_DIR
import metrics, inference
# NOT train.py or train_other_langs_v2.py - see the title cell for why this retrain
# needs its own training/eval logic.
from train_other_langs_v3 import train_stage_v3
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed
from datasets import Dataset
import pandas as pd

SEED = 42
set_seed(SEED)
MAX_LEN = 128
CHECKPOINT = "facebook/nllb-200-distilled-600M"   # raw base - same as v1/v2, genuinely independent
OTHER_LANGS_DATA_DIR = Path(DATA_DIR) / "other_langs"

LANGUAGE_CODES = {"kiswahili": "swh_Latn", "somali": "som_Latn", "dholuo": "luo_Latn"}
DOMAINS = ["Agriculture", "Health", "Education", "Security", "Governance"]

# Unchanged from v1/v2 - rejecting the reviewed suggestion's rank-32 / q+k+v+out_proj expansion.
LORA_KWARGS = {"r": 16, "lora_alpha": 32, "lora_dropout": 0.05}

OTHER_LANG_CFG_V3 = {
    "lora_target_modules": ["q_proj", "v_proj"],
    "freeze_layers": 6,
    "unfreeze_embeddings": False,
    "tgt_code": None,
    "related_code": None,
    "learning_rate": 1e-5,   # was 5e-5 in v1 - lowered given the demonstrated forgetting risk
    "batch_size": 32,
}

EVAL_STEPS = 100   # was 500 in v2 - checking more often to pin down the true early peak precisely
PATIENCE = 5       # was 3 in v2 - scaled up since checks are 5x more frequent now, so the
                   # stopping decision isn't twitchy on a single noisy check
MAX_EPOCHS = 5
MONITOR_SIZE = 150   # rows carved out of the clean 985-row shared_test pool for
                     # per-language monitoring - NOT shared_val (86.5% training overlap,
                     # confirmed) - see Section 4 for the actual split

print("configured language codes (verified against the real tokenizer next):", LANGUAGE_CODES)
print(f"learning_rate={OTHER_LANG_CFG_V3['learning_rate']}  eval_steps={EVAL_STEPS}  "
     f"patience={PATIENCE}  max_epochs={MAX_EPOCHS}  monitor_size={MONITOR_SIZE}")


project root: /home/jovyan/PSA/Final_Training
configured language codes (verified against the real tokenizer next): {'kiswahili': 'swh_Latn', 'somali': 'som_Latn', 'dholuo': 'luo_Latn'}
learning_rate=1e-05  eval_steps=100  patience=5  max_epochs=5  monitor_size=150


## 2. Verify the language codes against the real NLLB tokenizer

In [4]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
print("Checking each target language code against the real tokenizer:\n")
vocab = tokenizer.get_vocab()
verified = {}
for lang_name, code_guess in LANGUAGE_CODES.items():
    is_present = code_guess in vocab
    status = "CONFIRMED" if is_present else "NOT FOUND - do not trust this code"
    print(f"  {lang_name:12s} {code_guess:12s} {status}")
    verified[lang_name] = is_present


Checking each target language code against the real tokenizer:

  kiswahili    swh_Latn     CONFIRMED
  somali       som_Latn     CONFIRMED
  dholuo       luo_Latn     CONFIRMED


## 3. Load `05`'s three outputs (unchanged)

In [5]:
train_path = OTHER_LANGS_DATA_DIR / "other_langs_train.csv"
test_path = OTHER_LANGS_DATA_DIR / "other_langs_test.csv"
val_path = OTHER_LANGS_DATA_DIR / "other_langs_validation.csv"
for p in [train_path, test_path, val_path]:
    if not p.exists():
        raise FileNotFoundError(f"{p} not found - run 05_other_languages_preprocess.ipynb first.")

train_pool = pd.read_csv(train_path)
shared_test = pd.read_csv(test_path)
shared_val = pd.read_csv(val_path)
print(f"train pool: {len(train_pool):,} rows")
print(f"shared test set: {len(shared_test):,} rows")
print(f"shared validation set: {len(shared_val):,} rows")


train pool: 19,338 rows
shared test set: 985 rows
shared validation set: 1,589 rows


## 4. Split the clean pool, then build training records

`shared_test` (985 rows, confirmed only 5% near-duplicate overlap with the training
pool) is split ONCE, stratified by domain, into a **monitor** subset (used during
training for the per-language stopping decision) and a **final** subset (touched only
once, at the very end, in Section 8). Splitting once on the shared dataframe - not
per-language - keeps all three languages on the exact same underlying rows in each half,
preserving the "same content across languages" property the shared test set was built
for. `shared_val` is still used for `combined_val` (the Trainer's own loss-only logging,
never a stopping decision), but never again for anything chrF-based - that's what was
leaking (86.5% overlap with training, confirmed) in v2.

In [6]:
import random

def build_records(df, src_col, tgt_col, tgt_code, corpus_label, domain_col=None):
    out = []
    for r in df.itertuples(index=False):
        src, tgt = str(getattr(r, src_col)).strip(), str(getattr(r, tgt_col)).strip()
        if src and src.lower() != "nan" and tgt and tgt.lower() != "nan":
            rec = {"src_lang": "eng_Latn", "tgt_lang": tgt_code, "src": src, "tgt": tgt,
                  "corpus": corpus_label}
            if domain_col is not None:
                rec["domain"] = getattr(r, domain_col)
            out.append(rec)
    return out

_rng = random.Random(42)
_by_domain = {}
for _, row in shared_test.iterrows():
    _by_domain.setdefault(row["psa_domain"], []).append(row)

_monitor_idx, _final_idx = [], []
for _domain, _rows in _by_domain.items():
    _shuffled = _rows[:]
    _rng.shuffle(_shuffled)
    _n_monitor = round(len(_shuffled) * MONITOR_SIZE / len(shared_test))
    _monitor_idx.extend(r.name for r in _shuffled[:_n_monitor])
    _final_idx.extend(r.name for r in _shuffled[_n_monitor:])

shared_test_monitor = shared_test.loc[_monitor_idx].reset_index(drop=True)
shared_test_final = shared_test.loc[_final_idx].reset_index(drop=True)
print(f"shared_test split: monitor={len(shared_test_monitor)} rows, final={len(shared_test_final)} rows")
print("monitor domain balance:", shared_test_monitor["psa_domain"].value_counts().to_dict())
print("final domain balance:  ", shared_test_final["psa_domain"].value_counts().to_dict())

records = {}
for lang in ["kiswahili", "somali", "dholuo"]:
    if not verified.get(lang, False):
        print(f"skipping {lang} - code not verified (Section 2)")
        continue
    target_col = lang.capitalize()
    train_rows = build_records(train_pool, "English", target_col, LANGUAGE_CODES[lang], f"{lang}_target")
    val_rows = build_records(shared_val, "English", target_col, LANGUAGE_CODES[lang], f"{lang}_target")
    monitor_rows = build_records(shared_test_monitor, "English", target_col, LANGUAGE_CODES[lang], f"{lang}_target",
                                 domain_col="psa_domain")
    final_rows = build_records(shared_test_final, "English", target_col, LANGUAGE_CODES[lang], f"{lang}_target",
                               domain_col="psa_domain")
    records[lang] = (train_rows, val_rows, monitor_rows, final_rows)
    print(f"{lang}_target: train={len(train_rows):,} | val={len(val_rows):,} | "
         f"monitor={len(monitor_rows):,} | final={len(final_rows):,}")

combined_train = [r for lang in records for r in records[lang][0]]
combined_val = [r for lang in records for r in records[lang][1]]
monitor_by_lang = {lang: records[lang][2] for lang in records}   # clean - for PerLanguageProtectiveEval
final_by_lang = {lang: records[lang][3] for lang in records}      # clean - for the real, once-only evaluation

print(f"\ncombined train (all languages): {len(combined_train):,} rows")
print(f"combined validation (loss-tracking only, NOT used for stopping): {len(combined_val):,} rows")


shared_test split: monitor=150 rows, final=835 rows
monitor domain balance: {'Health': 60, 'Agriculture': 40, 'Education': 23, 'Governance': 17, 'Security': 10}
final domain balance:   {'Health': 334, 'Agriculture': 222, 'Education': 129, 'Governance': 94, 'Security': 56}
kiswahili_target: train=19,338 | val=1,589 | monitor=150 | final=835
somali_target: train=19,338 | val=1,589 | monitor=150 | final=835
dholuo_target: train=19,338 | val=1,589 | monitor=150 | final=835

combined train (all languages): 58,014 rows
combined validation (loss-tracking only, NOT used for stopping): 4,767 rows


## 5. Tokenization - combined only

No per-language tokenization this time - independent training isn't run in this
notebook, so only the combined tensors are needed for `Trainer`. `monitor_by_lang` and
`final_by_lang` above stay as raw text rows; `PerLanguageProtectiveEval` tokenizes on the
fly per generation call, matching Section 8/`inference.py`'s own pattern exactly.

In [7]:
def nllb_preprocess_combined(batch):
    all_ids = []
    for sl, tl, src, tgt in zip(batch["src_lang"], batch["tgt_lang"], batch["src"], batch["tgt"]):
        tokenizer.src_lang = sl
        tokenizer.tgt_lang = tl
        enc = tokenizer(src, text_target=tgt, max_length=MAX_LEN, truncation=True)
        all_ids.append(enc)
    return {"input_ids": [e["input_ids"] for e in all_ids],
            "attention_mask": [e["attention_mask"] for e in all_ids],
            "labels": [e["labels"] for e in all_ids]}

def tokenize_combined(rows):
    ds = Dataset.from_list(rows)
    return ds.map(nllb_preprocess_combined, batched=True, batch_size=16)

tok_combined_train = tokenize_combined(combined_train)
tok_combined_val = tokenize_combined(combined_val)
print(f"combined tokenization complete - {len(tok_combined_train)} train, {len(tok_combined_val)} val")


Map:   0%|          | 0/58014 [00:00<?, ? examples/s]

Map:   0%|          | 0/4767 [00:00<?, ? examples/s]

combined tokenization complete - 58014 train, 4767 val


## 6. Zero-shot baseline (unchanged, shared with v1/v2)

In [8]:
zeroshot_path = f"{CHECKPOINTS_DIR}/nllb_other_langs_zeroshot"
if not Path(zeroshot_path).exists():
    AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT).save_pretrained(zeroshot_path, safe_serialization=True)
    tokenizer.save_pretrained(zeroshot_path)
    print(f"zero-shot baseline saved: {zeroshot_path}")
else:
    print(f"zero-shot baseline already present (from an earlier run): {zeroshot_path}")


zero-shot baseline already present (from an earlier run): checkpoints/nllb_other_langs_zeroshot


## 7. Train the combined model - v3 recipe

One call to `train_stage_v3()` - the gentler recipe (v2), on a clean, non-leaked
per-language monitoring split (v3), checking every `EVAL_STEPS` steps, stops when Somali
or Dholuo stops improving. See `train_other_langs_v3.py` for the full implementation.

In [9]:
combined_path_v3, train_seconds, per_lang_cb = train_stage_v3(
    run_id="nllb_combined_other_langs_v3",
    cfg=OTHER_LANG_CFG_V3,
    stage_name="combined_other_langs_v3",
    init_from=CHECKPOINT,
    train_tok=tok_combined_train,
    val_tok=tok_combined_val,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    lora_kwargs=LORA_KWARGS,
    val_by_lang=monitor_by_lang,
    language_codes=LANGUAGE_CODES,
    eval_steps=EVAL_STEPS,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    checkpoints_dir=CHECKPOINTS_DIR,
    logs_dir=LOGS_DIR,
)
print(f"\ncombined_v3 checkpoint: {combined_path_v3}")
print(f"per-language best scores during training: {per_lang_cb.best}")


  disabled the cuDNN SDPA backend (known crash risk on this environment) - flash-attention/memory-efficient attention remain available
  nllb_combined_other_langs_v3   arch=nllb  stage=combined_other_langs_v3  init=facebook/nllb-200-distilled-600M  use_lora=True  [v3: clean monitoring split]
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_combined_other_langs_v3
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db
  lr=1e-05  eval_steps=100  patience=5  max_epochs=5


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821
  (this transformers version has no warmup_ratio - using warmup_steps=362 instead, ~4% of 9060 total steps)
  schedule -> {'lr_scheduler_type': 'linear', 'warmup_steps': 362}


Step,Training Loss,Validation Loss
100,1.027032,0.745294
200,1.018826,0.737454
300,1.005336,0.702834
400,0.911780,0.639352
500,0.888880,0.584169
600,0.822802,0.542818


perlang_eval_kiswahili_step100      chrF2++=80.96  BLEU=69.73
perlang_eval_somali_step100         chrF2++=86.89  BLEU=77.64
perlang_eval_dholuo_step100         chrF2++=76.02  BLEU=65.66
[per-language eval @ step 100] kiswahili=80.96 | somali=86.89 | dholuo=76.02
perlang_eval_kiswahili_step200      chrF2++=81.27  BLEU=70.12
perlang_eval_somali_step200         chrF2++=85.88  BLEU=75.54
perlang_eval_dholuo_step200         chrF2++=74.85  BLEU=63.97
[per-language eval @ step 200] kiswahili=81.27 | somali=85.88 | dholuo=74.85
perlang_eval_kiswahili_step300      chrF2++=81.05  BLEU=69.14
perlang_eval_somali_step300         chrF2++=83.23  BLEU=70.88
perlang_eval_dholuo_step300         chrF2++=72.37  BLEU=60.52
[per-language eval @ step 300] kiswahili=81.05 | somali=83.23 | dholuo=72.37
perlang_eval_kiswahili_step400      chrF2++=80.33  BLEU=68.10
perlang_eval_somali_step400         chrF2++=80.58  BLEU=65.09
perlang_eval_dholuo_step400         chrF2++=70.58  BLEU=57.28
[per-language eval @ step

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


combined_v3 checkpoint: checkpoints/nllb_combined_other_langs_v3/merged
per-language best scores during training: {'somali': 86.88655729593509, 'dholuo': 76.01688349010331}


## 8. Evaluate: zero-shot vs. v3 combined, on the clean FINAL split only

Full generation (beam=4, correct `forced_bos_token_id`, anti-repetition guards) over
`final_by_lang` - the subset of the clean 985-row pool never touched during training
(not `monitor_by_lang`, and not the full 985 - `monitor_by_lang` was already used to pick
the checkpoint, so scoring it again here wouldn't be a fair, once-only final read). No
v1/v2 comparison - neither is re-evaluated or referenced here.

In [10]:
def load_and_generate(path, tgt_code):
    model = AutoModelForSeq2SeqLM.from_pretrained(path)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    tok_ = AutoTokenizer.from_pretrained(path)
    def generate(text):
        tok_.src_lang = "eng_Latn"
        enc = tok_(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
        forced_bos = tok_.convert_tokens_to_ids(tgt_code)
        out = model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LEN,
                             no_repeat_ngram_size=3, repetition_penalty=1.2)
        return tok_.decode(out[0], skip_special_tokens=True)
    return model, generate

eval_results = []
for lang, test_rows in final_by_lang.items():
    tgt_code = LANGUAGE_CODES[lang]

    for run_name, path in [("zeroshot", zeroshot_path), ("v3_combined", combined_path_v3)]:
        model, generate = load_and_generate(path, tgt_code)

        preds = [generate(r["src"]) for r in test_rows]
        score = metrics.score(preds, [r["tgt"] for r in test_rows], f"nllb_{run_name}_{lang}_overall")
        score.update({"language": lang, "run": run_name, "domain": "ALL", "n": len(test_rows)})
        eval_results.append(score)

        for domain in DOMAINS:
            subset = [r for r in test_rows if r.get("domain") == domain]
            if not subset:
                continue
            preds_d = [generate(r["src"]) for r in subset]
            score_d = metrics.score(preds_d, [r["tgt"] for r in subset], f"nllb_{run_name}_{lang}_{domain}")
            score_d.update({"language": lang, "run": run_name, "domain": domain, "n": len(subset)})
            eval_results.append(score_d)

        del model
        torch.cuda.empty_cache()

results_df = pd.DataFrame(eval_results)[["language", "run", "domain", "n", "chrf2pp", "bleu"]]
results_df.to_csv(f"{LOGS_DIR}/nllb_other_languages_v3_results.csv", index=False)

print("\n=== Zero-shot vs. v3 combined, overall (ALL domains) chrF2++, on the CLEAN final split ===")
overall = results_df[results_df["domain"] == "ALL"].pivot_table(index="language", columns="run", values="chrf2pp")
overall = overall[["zeroshot", "v3_combined"]]
overall["v3_vs_zeroshot"] = overall["v3_combined"] - overall["zeroshot"]
print(overall.round(2).to_string())

print("\n=== v3 combined, chrF2++ by domain ===")
by_domain = results_df[(results_df["domain"] != "ALL") & (results_df["run"] == "v3_combined")]
print(by_domain.pivot_table(index="language", columns="domain", values="chrf2pp").round(1).to_string())


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_zeroshot_kiswahili_overall     chrF2++=72.65  BLEU=54.66
nllb_zeroshot_kiswahili_Agriculture chrF2++=73.40  BLEU=55.01
nllb_zeroshot_kiswahili_Health      chrF2++=70.97  BLEU=52.41
nllb_zeroshot_kiswahili_Education   chrF2++=77.88  BLEU=63.17
nllb_zeroshot_kiswahili_Security    chrF2++=75.68  BLEU=58.05
nllb_zeroshot_kiswahili_Governance  chrF2++=70.48  BLEU=51.29


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_v3_combined_kiswahili_overall  chrF2++=72.72  BLEU=54.74
nllb_v3_combined_kiswahili_Agriculture chrF2++=73.40  BLEU=55.01
nllb_v3_combined_kiswahili_Health   chrF2++=70.98  BLEU=52.46
nllb_v3_combined_kiswahili_Education chrF2++=78.16  BLEU=63.44
nllb_v3_combined_kiswahili_Security chrF2++=75.68  BLEU=58.05
nllb_v3_combined_kiswahili_Governance chrF2++=70.71  BLEU=51.54


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

...

nllb_v3_combined_somali_overall     chrF2++=73.69  BLEU=52.10
nllb_v3_combined_somali_Agriculture chrF2++=71.26  BLEU=49.43
nllb_v3_combined_somali_Health      chrF2++=72.12  BLEU=49.33
nllb_v3_combined_somali_Education   chrF2++=78.04  BLEU=58.79
nllb_v3_combined_somali_Security    chrF2++=77.83  BLEU=59.65
nllb_v3_combined_somali_Governance  chrF2++=78.13  BLEU=58.00


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_zeroshot_dholuo_overall        chrF2++=60.11  BLEU=41.86
nllb_zeroshot_dholuo_Agriculture    chrF2++=65.32  BLEU=50.95
nllb_zeroshot_dholuo_Health         chrF2++=56.41  BLEU=36.07
nllb_zeroshot_dholuo_Education      chrF2++=59.94  BLEU=40.25
nllb_zeroshot_dholuo_Security       chrF2++=57.28  BLEU=35.77
nllb_zeroshot_dholuo_Governance     chrF2++=62.51  BLEU=38.92


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_v3_combined_dholuo_overall     chrF2++=60.20  BLEU=41.95
nllb_v3_combined_dholuo_Agriculture chrF2++=65.36  BLEU=50.95
nllb_v3_combined_dholuo_Health      chrF2++=56.38  BLEU=36.08
nllb_v3_combined_dholuo_Education   chrF2++=60.20  BLEU=40.70
nllb_v3_combined_dholuo_Security    chrF2++=57.72  BLEU=36.16
nllb_v3_combined_dholuo_Governance  chrF2++=62.75  BLEU=39.16

=== Zero-shot vs. v3 combined, overall (ALL domains) chrF2++, on the CLEAN final split ===
run        zeroshot  v3_combined  v3_vs_zeroshot
language                                        
dholuo        60.11        60.20            0.09
kiswahili     72.65        72.72            0.07
somali        73.69        73.69            0.00

=== v3 combined, chrF2++ by domain ===
domain     Agriculture  Education  Governance  Health  Security
language                                                       
dholuo            65.4       60.2        62.8    56.4      57.7
kiswahili         73.4       78.2        70.7    71.0      

## 9. Save deployment-ready checkpoint

In [11]:
inference.save_for_deployment(combined_path_v3, "nllb", "nllb_combined_other_langs_v3", deployment_dir=DEPLOYMENT_DIR)


deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_combined_other_langs_v3  (1 safetensors file(s))


PosixPath('deployment/nllb/nllb_combined_other_langs_v3')

## 10. Demonstration

In [12]:
samples = [
    "Farmers are urged to prioritize safe agrochemical usage this season.",
    "Ministry of Health: report suspected cholera cases immediately.",
]

demo_model = AutoModelForSeq2SeqLM.from_pretrained(combined_path_v3)
demo_model.to("cuda" if torch.cuda.is_available() else "cpu")
demo_tok = AutoTokenizer.from_pretrained(combined_path_v3)

for lang in records:
    demo_tok.src_lang = "eng_Latn"
    forced_bos = demo_tok.convert_tokens_to_ids(LANGUAGE_CODES[lang])

    print(f"NLLB (combined v3) - English to {lang.capitalize()} (PSA-adapted)\n")
    for text in samples:
        enc = demo_tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(demo_model.device)
        out = demo_model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LEN,
                                  no_repeat_ngram_size=3, repetition_penalty=1.2)
        print(f"[en] {text}")
        print("  ->", demo_tok.decode(out[0], skip_special_tokens=True))
        print()
del demo_model
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

NLLB (combined v3) - English to Kiswahili (PSA-adapted)

[en] Farmers are urged to prioritize safe agrochemical usage this season.
  -> Wakulima wanahimizwa waweke kipaumbele matumizi salama ya kemikali za kilimo msimu huu.

[en] Ministry of Health: report suspected cholera cases immediately.
  -> Wizara ya Afya: kuripoti visa vya kuaminika kwa ugonjwa wa cholera mara moja.

NLLB (combined v3) - English to Somali (PSA-adapted)

[en] Farmers are urged to prioritize safe agrochemical usage this season.
  -> Beeraleyda waxaa lagu boorinayaa inay mudnaanta siiyaan isticmaalka kiimikada beeraha ee ammaanka ah xilli ciyaareedkan.

[en] Ministry of Health: report suspected cholera cases immediately.
  -> Wasaaradda Caafimaadka: si degdeg ah u soo sheeg kiisaska laga shakiyo in ay ka jiraan cudurka kolera.

NLLB (combined v3) - English to Dholuo (PSA-adapted)

[en] Farmers are urged to prioritize safe agrochemical usage this season.
  -> Ojiwo jopur mondo oket mokwongo tiyo gi gik ma nigi kemi